In [0]:
%pip install apache-sedona

In [0]:
dbutils.library.restartPython()

In [0]:
from sedona.spark import SedonaContext
sedona = SedonaContext.create(spark)

In [0]:
from pyspark.sql import functions as F

fhv = spark.table("silver.fhv_taxi_trips").alias("fhv")
yellow = spark.table("silver.yellow_taxi_trips").alias("y")
green = spark.table("silver.green_taxi_trips").alias("g")
hvfhv = spark.table("silver.high_volume_fhv").alias("h")

In [0]:
#target (combined dataframe) schema
target_schema = {
    "taxi_type": "string",
    "service_type": "string",
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "trip_time": "double",
    "dolocationid": "int",
    "pulocationid": "int",
    "trip_distance": "double",
    "extra": "decimal(10,2)",
    "fare_amount": "decimal(10,2)",
    "mta_tax": "decimal(10,2)",
    "tip_amount": "decimal(10,2)",
    "tolls_amount": "decimal(10,2)",
    "total_amount": "decimal(10,2)",
    "payment_types": "string",
    "rate_codes": "string",
    "vendors": "string",
    "base_passenger_fare": "decimal(10,2)",
    "bcf": "decimal(10,2)",
}

#match the tables to schema
def align(df):
    return df.select(*[
        (F.col(c).cast(t) if c in df.columns else F.lit(None).cast(t)).alias(c)
        for c, t in target_schema.items()
    ])

#combine
combined_df = (
    align(hvfhv)
    .unionByName(align(yellow))
    .unionByName(align(green))
    .unionByName(align(fhv))
)

In [0]:
combined_df.createOrReplaceTempView("combined_df")

In [0]:
%sql
MERGE INTO group3_gp.gold.tabel_count AS target
USING (
    SELECT 
        'Combined' AS source,
        COUNT(*) AS row_count,
        'Combine' AS reason
    FROM combined_df
) AS src
ON target.source = src.source

WHEN MATCHED THEN
    UPDATE SET 
        target.row_count = src.row_count,
        target.reason = src.reason

WHEN NOT MATCHED THEN
    INSERT (source, row_count, reason)
    VALUES (src.source, src.row_count, src.reason);

adding nulls instead of faulty values
also adding some columns that show if distance, time or total amount was calculated, they are true if we did not have a value to start with

In [0]:

#-------------------------------------------------------------------------------------------------------------------
#creating df with clean columns
df = (
    combined_df
    #establishing nulls rather than unknown or 0 values
    .withColumn(
        "dolocationid",
        F.when((F.col("dolocationid") <= 0) | (F.col("dolocationid") > 265), F.lit(None))
        .otherwise(F.col("dolocationid"))
    )
    .withColumn(
        "pulocationid",
        F.when((F.col("pulocationid") <= 0) | (F.col("pulocationid") > 265), F.lit(None))
        .otherwise(F.col("pulocationid"))
    )
    .withColumn(
        "payment_types",
        F.when(F.col("payment_types").isin("Unknown", "Unrecognised"), F.lit(None))
        .otherwise(F.col("payment_types"))
    )
    .withColumn(
        "rate_codes",
        F.when(F.col("rate_codes").isin("Unknown", "Unrecognised", "Group ride"), F.lit(None))
        .otherwise(F.col("rate_codes"))
    )
    .withColumn(
        "vendors",
        F.when(F.col("vendors").isin("Unknown", "Unrecognised"), F.lit(None))
        .otherwise(F.col("vendors"))
    )
#-------------------------------------------------------------------------------------------------------------------
    #calculating trip distance if it is missing and saving the fact that it was calculated if it is
    .withColumn(
        "calculated_trip_distance",
        (F.col("trip_distance") <= 0.0) | (F.col("trip_distance").isNull())
    )
    .withColumn(
        "trip_distance",
        F.when(
            F.col("calculated_trip_distance"),
            F.lit(None)
        ).otherwise(F.col("trip_distance"))
    )
#-------------------------------------------------------------------------------------------------------------------
    #calculating trip time if it is missing and saving the fact that it was calculated if it is
    .withColumn(
        "calculated_trip_time",
        (F.col("trip_time").isNull()) | (F.col("trip_time") == 0.0)
    )
    .withColumn(
        "trip_time",
        F.when(
            F.col("calculated_trip_time"),
            F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")
        ).otherwise(F.col("trip_time"))
    )
    .withColumn(
        "bcf",
        F.when(
            (F.col("bcf").isNull()),
            F.lit(None)
        ).otherwise(F.col("bcf"))
    )
    .withColumn(
        "extra",
        F.when(
            (F.col("extra").isNull()),
            F.lit(None)
        ).otherwise(F.col("extra"))
    )
    .withColumn(
        "mta_tax",
        F.when(
            (F.col("mta_tax").isNull()),
            F.lit(None)
        ).otherwise(F.col("mta_tax"))
    )
    .withColumn(
        "tip_amount",
        F.when(
            (F.col("tip_amount").isNull()),
            F.lit(None)
        ).otherwise(F.col("tip_amount"))
    )
    .withColumn(
        "tolls_amount",
        F.when(
            (F.col("tolls_amount").isNull()),
            F.lit(None)
        ).otherwise(F.col("tolls_amount"))
    )
    .withColumn(
        "base_passenger_fare",
        F.when(
            ((F.col("base_passenger_fare").isNull()) | (F.col("base_passenger_fare") <= 0.0)),
            F.lit(None)
        ).otherwise(F.col("base_passenger_fare"))
    )
    .withColumn(
        "fare_amount",
        F.when(
            ((F.col("fare_amount").isNull()) | (F.col("fare_amount") <= 0.0)),
            F.lit(None)
        ).otherwise(F.col("fare_amount"))
    )
    .withColumn(
        "calculated_total_amount",
        (F.col("total_amount").isNull()) | (F.col("total_amount") <= 0.0)
    )
    .withColumn(
        "total_amount",
        F.when(
            F.col("calculated_total_amount"),
            F.greatest(
                (
                    F.coalesce(F.col("base_passenger_fare"), F.lit(0.0)) +
                    F.coalesce(F.col("tolls_amount"), F.lit(0.0)) +
                    F.coalesce(F.col("tip_amount"), F.lit(0.0)) +
                    F.coalesce(F.col("extra"), F.lit(0.0)) +
                    F.coalesce(F.col("mta_tax"), F.lit(0.0)) +
                    F.coalesce(F.col("bcf"), F.lit(0.0))
                ),
                (
                    F.coalesce(F.col("fare_amount"), F.lit(0.0)) +
                    F.coalesce(F.col("tolls_amount"), F.lit(0.0)) +
                    F.coalesce(F.col("tip_amount"), F.lit(0.0)) +
                    F.coalesce(F.col("extra"), F.lit(0.0)) +
                    F.coalesce(F.col("mta_tax"), F.lit(0.0)) +
                    F.coalesce(F.col("bcf"), F.lit(0.0))
                )
            )
        ).otherwise(F.col("total_amount"))
    )
    .withColumn(
        "total_amount",
        F.when(
            (F.col("total_amount").isNull()) | (F.col("total_amount") <= 0.0),
            F.lit(None)
        ).otherwise(F.col("total_amount"))
    )
#-------------------------------------------------------------------------------------------------------------------
    #drop rows where pickup and dropoff location is missing
    .filter(
        F.col("pulocationid").isNotNull() |
        F.col("dolocationid").isNotNull()
    )
)

Calculating distance between locationid's using sedona and dim_zone table

In [0]:
# geometry_wkt is a WKT string.
# ST_GeomFromWKT converts it to a geometry.
# st_setsrid(..., 2263) assigns coordinates reference system (EPSG:2263).
# ST_Centroid computes the geometry centroid.
# ST_Transform(..., 4326) converts to lat/lon (EPSG:4326).
# ST_X / ST_Y extract longitude and latitude.
zones_centroids = spark.sql("""
SELECT
  zone_id AS locationid,
  ST_X(ST_Transform(ST_Centroid(st_setsrid(ST_GeomFromWKT(geometry_wkt), 2263)), 4326)) AS lon,
  ST_Y(ST_Transform(ST_Centroid(st_setsrid(ST_GeomFromWKT(geometry_wkt), 2263)), 4326)) AS lat
FROM group3_gp.gold.dim_zone
WHERE geometry_wkt IS NOT NULL
""")
#join into df
df = df \
    .join(zones_centroids.withColumnRenamed("locationid", "pulocationid"), "pulocationid", "left") \
    .withColumnRenamed("lon", "pu_lon").withColumnRenamed("lat", "pu_lat") \
    .join(zones_centroids.withColumnRenamed("locationid", "dolocationid"), "dolocationid", "left") \
    .withColumnRenamed("lon", "do_lon").withColumnRenamed("lat", "do_lat")
#calculate using Haversine formula in miles and store in new column
df = df.withColumn(
    "calc_distance",
    3958.8 * 2 * F.asin(F.sqrt(
        F.pow(F.sin((F.radians(F.col("do_lat")) - F.radians(F.col("pu_lat"))) / 2), 2) +
        F.cos(F.radians(F.col("pu_lat"))) *
        F.cos(F.radians(F.col("do_lat"))) *
        F.pow(F.sin((F.radians(F.col("do_lon")) - F.radians(F.col("pu_lon"))) / 2), 2)
    ))
)

Setting trip_distance to calculated calc_distance if it needs to be calculated (which we checked above if it was needed)
Also dropping columns no longer needed

In [0]:
final_df = df.withColumn(
    "trip_distance",
    F.when(
        F.col("calculated_trip_distance"),
        F.col("calc_distance")
    ).otherwise(F.col("trip_distance"))
).drop("pu_lon", "pu_lat", "do_lon", "do_lat", "calc_distance", "base_passenger_fare", "fare_amount")

Create tempview to load rowcount into tabel where we can keep track of development

In [0]:
final_df = final_df.dropDuplicates()
dup_keys = [
    "taxi_type",
    "pickup_datetime",
    "dropoff_datetime",
    "pulocationid",
    "dolocationid",
    "trip_distance",
    "total_amount"
]
final_df = final_df.dropDuplicates(dup_keys)

In [0]:
final_df.createOrReplaceTempView("final_df")

In [0]:
%sql
MERGE INTO group3_gp.gold.tabel_count AS target
USING (
    SELECT 
        'Combined_Clean' AS source,
        COUNT(*) AS row_count,
        'After filtering, dropping obviously invalid values, missing geodata, and duplicates' AS reason
    FROM final_df
) AS src
ON target.source = src.source

WHEN MATCHED THEN
    UPDATE SET 
        target.row_count = src.row_count,
        target.reason = src.reason

WHEN NOT MATCHED THEN
    INSERT (source, row_count, reason)
    VALUES (src.source, src.row_count, src.reason);

spara till ett combined silver table

In [0]:
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.silver.combined_taxi_trips")

Create company dimension based on silver combined table

In [0]:
%sql
SELECT * FROM gold.tabel_count